# [센서 실습 20] 진동 센서 – ADXL345 3축 가속도 센서

**ADXL345**로 모터의 진동을 400Hz로 측정해 0.5초마다 **진동 실효값(RMS)** 을 계산하고, **FFT(주파수 분석)** 로 정상·이상 진동을 비교하는 **예지 보전** 실습입니다.

**이 노트북의 흐름**
1. 센서의 원리와 부품 사양
2. 라즈베리파이 연결 방법
3. 실제 센서를 읽는 코드 (라즈베리파이용)
4. 장비가 없을 때 쓰는 **가상 센서** (같은 코드로 실행)
5. 수집 → 판정 → 시각화 → 저장·IoT 전송

> 장비가 없어도 `USE_REAL_SENSOR = False` 상태 그대로 위에서부터 `Shift + Enter`로 모두 실행됩니다.

## 1. 센서의 원리
공장의 모터·펌프는 고장 나기 전에 **진동이 먼저 커지거나 모양이 변합니다.** 이를 미리 알아내 정비하는 것이 **예지 보전(Predictive Maintenance)** 입니다.
- 진동 센서 = 가속도 센서를 기계에 붙여 빠르게 측정
- **RMS(실효값)**: 진동의 평균적인 세기 $\sqrt{\frac{1}{N}\sum a^2}$
- **FFT**: 진동을 주파수 성분으로 나눔
  - 회전 주파수(1×)가 커지면 → **불균형**
  - 높은 주파수 성분이 생기면 → **베어링 손상** 가능성
- 1800rpm 모터의 회전 주파수 = 1800 ÷ 60 = **30Hz**
- 나이퀴스트 정리: 측정 주파수는 보려는 최고 주파수의 2배 이상이어야 함 (400Hz 측정 → 200Hz까지)

### 부품 사양
| 항목 | 내용 |
|---|---|
| 범위 | ±2 / 4 / 8 / 16 g |
| 분해능 | 최대 13비트 (3.9 mg/LSB) |
| 출력 속도 | 최대 3200 Hz |
| 전원 | 2.0 ~ 3.6V |
| 통신 | I2C 주소 0x53 (또는 SPI) |

## 2. 라즈베리파이 연결 방법
**준비물**: ADXL345 모듈(GY-291), 점퍼선 4개, (실험용) 소형 DC 모터

| 연결할 곳 | 라즈베리파이 | 설명 |
|---|---|---|
| ADXL345 VCC | 3.3V (1번 핀) |  |
| ADXL345 GND | GND (6번 핀) |  |
| ADXL345 SCL | GPIO3 / SCL (5번 핀) | I2C 클록 |
| ADXL345 SDA | GPIO2 / SDA (3번 핀) | I2C 데이터 (주소 0x53) |

센서는 기계에 **단단히 고정**(나사·순간접착제·자석)해야 진동이 그대로 전달됩니다. 양면테이프는 높은 주파수를 흡수해 버립니다.

**인터페이스 활성화**
- **I2C**: `sudo raspi-config` → Interface Options → **I2C** → Enable, 연결 확인: `i2cdetect -y 1`

**라이브러리 설치 (라즈베리파이 터미널)**
```bash
pip3 install adafruit-circuitpython-adxl34x
```

## 3. 공통 준비 – 한글 폰트와 라이브러리

In [ ]:
# Colab에서 그래프의 한글이 깨지지 않도록 나눔고딕 폰트를 설정합니다. (처음 한 번 실행)
import os, subprocess
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

font_path = "/usr/share/fonts/truetype/nanum/NanumGothic.ttf"
if not os.path.exists(font_path):
    subprocess.run("apt-get -qq -y install fonts-nanum", shell=True,
                   stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
if os.path.exists(font_path):
    fm.fontManager.addfont(font_path)
    plt.rc("font", family="NanumGothic")
plt.rcParams["axes.unicode_minus"] = False
print("한글 폰트 설정 완료")

In [ ]:
import time, json
from datetime import datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## 4. 실제 센서 읽기 (라즈베리파이용)
### 4-1. 원시값 → 물리량 변환식 (실제·가상 공통)
여러 샘플을 모아 **RMS**를 구하고, **FFT**로 주파수 스펙트럼을 구하는 공통 함수입니다.

In [ ]:
def window_rms(values):
    v = np.asarray(values) - np.mean(values)            # 중력 등 직류 성분 제거
    return float(np.sqrt(np.mean(v ** 2)))

def spectrum(values, fs):
    v = np.asarray(values) - np.mean(values)
    amp = np.abs(np.fft.rfft(v * np.hanning(len(v)))) * 2 / (len(v) * 0.5)
    freq = np.fft.rfftfreq(len(v), 1 / fs)
    return freq, amp

### 4-2. 실제 센서 클래스
Adafruit 라이브러리로 출력 속도를 800Hz로 설정하고 Z축 가속도를 읽습니다. (파이썬 반복문의 한계로 실제 수집 속도는 수백 Hz 수준입니다.)

> 이 셀은 **클래스를 정의만** 합니다. 하드웨어 라이브러리는 객체를 만들 때(`__init__`) 불러오므로 Colab에서 실행해도 오류가 나지 않습니다. 실제 동작은 라즈베리파이에서 `USE_REAL_SENSOR = True`로 해야 합니다.

In [ ]:
class RealADXL345:
    def __init__(self):
        import board, adafruit_adxl34x
        self.dev = adafruit_adxl34x.ADXL345(board.I2C())
        self.dev.range = adafruit_adxl34x.Range.RANGE_4_G
        self.dev.data_rate = adafruit_adxl34x.DataRate.RATE_800_HZ

    def read(self):
        x, y, z = self.dev.acceleration                    # m/s²
        return {"az_g": round(z / 9.80665, 4)}

## 5. 장비가 없을 때 – 가상 센서
### 왜 가상 센서를 쓰나요?
실제 센서의 값은 다음 순서로 만들어집니다.

```
① 전기 신호 (전압, 펄스, 0/1)  →  ② 원시값 raw (ADC 숫자, 바이트, 문자열)  →  ③ 변환식  →  ④ 물리량 (°C, ppm, cm …)
```
가상 센서는 ①을 건너뛰고 **②원시값을 실제 센서와 똑같은 형식으로** 만들어 줍니다.
따라서 ③변환식과 그 뒤의 판정·시각화·저장 코드는 실제 장비와 가상 장비에서 **완전히 같은 코드**를 사용합니다.

```
[장비 있음]  센서 ──배선──▶ 라즈베리파이 ──▶ Real….read()  ─┐
                                                             ├──▶ 변환식 → 판정 → 그래프 → 저장/전송
[장비 없음]  가정한 상황(시나리오) ──▶ Virtual….read() ──────┘
```
나중에 장비를 연결하면 `USE_REAL_SENSOR = True` **한 줄만** 바꾸면 됩니다.

### 가상 값은 이렇게 정했습니다
| 구간 | 가정한 상태 | 진동 성분 |
|---|---|---|
| 0 ~ 10초 | 정상 운전 (1800rpm) | 30Hz 0.08g + 60Hz 0.03g |
| 10 ~ 20초 | 불균형 진행 + 베어링 손상 시작 | 30Hz가 0.08 → 0.35g로 증가, 120Hz 성분 0.15g 발생 |
| 항상 | 측정 잡음 | ±0.02g |
| 센서 출력 | 3.9mg 단위 정수 (13비트) | ADXL345 분해능 |

In [ ]:
class VirtualADXL345:
    LSB = 0.0039                                           # 3.9 mg/LSB

    def __init__(self, dt=0.0025, seed=20):
        self.dt, self.t = dt, 0.0
        self.rng = np.random.default_rng(seed)

    def read(self):
        t = self.t
        k = 0.0 if t < 10 else min(1.0, (t - 10) / 5)      # 고장 진행 정도
        a = (0.08 + 0.27 * k) * np.sin(2 * np.pi * 30 * t) + 0.03 * np.sin(2 * np.pi * 60 * t)
        a += 0.15 * k * np.sin(2 * np.pi * 120 * t) + self.rng.normal(0, 0.02)
        raw = int(round((1.0 + a) / self.LSB))             # 중력 1g + 진동, 정수 출력
        self.t += self.dt
        return {"az_g": round(raw * self.LSB, 4)}

In [ ]:
# 가상 센서 동작 확인 – read()를 3번 호출
test = VirtualADXL345(dt=0.0025)
for _ in range(3):
    print(test.read())

## 6. 센서 선택과 데이터 수집
400Hz(0.0025초 간격)로 20초 동안 측정합니다.

- 실제 장비는 `time.sleep(INTERVAL)`으로 측정 간격만큼 기다립니다.
- 가상 센서는 기다리지 않고 시간을 건너뛰므로, 몇 시간·며칠 분량의 데이터도 바로 만들어집니다.

In [ ]:
USE_REAL_SENSOR = False      # ← 라즈베리파이에 센서를 연결했다면 True 로 변경
N_SAMPLES = 8000               # 측정 횟수
INTERVAL = 0.0025               # 측정 간격(초)

sensor = RealADXL345() if USE_REAL_SENSOR else VirtualADXL345(dt=INTERVAL)

records = []
for i in range(N_SAMPLES):
    value = sensor.read()
    records.append({"time_s": round(i * INTERVAL, 4), **value})
    if USE_REAL_SENSOR:
        time.sleep(INTERVAL)

df = pd.DataFrame(records)
print(f"{len(df)}개 측정 완료 (총 {N_SAMPLES * INTERVAL / 60:.1f}분 분량)")
df.head()

In [ ]:
df.describe().round(3)

## 7. 판정 – 이벤트 감지
0.5초(200개)씩 끊어 RMS를 구하고, **0.15g 초과 시 이상 진동**으로 판정합니다. 정상 구간과 이상 구간의 FFT도 비교합니다.

In [ ]:
FS, WIN, TH = 1 / INTERVAL, 200, 0.15
rows = []
for i in range(0, len(df) - WIN + 1, WIN):
    seg = df["az_g"].values[i:i + WIN]
    rows.append({"time_s": df["time_s"].iloc[i], "rms_g": window_rms(seg)})
rms = pd.DataFrame(rows)
rms["alarm"] = rms["rms_g"] > TH
first = rms[rms["alarm"]]
print(f"정상 구간 평균 RMS: {rms[rms.time_s < 10].rms_g.mean():.3f} g")
if len(first): print(f"⚠️ 이상 진동 최초 감지: {first.time_s.iloc[0]:.1f}초 (RMS {first.rms_g.iloc[0]:.3f} g) → 정비 요청")

f_n, a_n = spectrum(df[df.time_s < 5]["az_g"].values[:1600], FS)
f_f, a_f = spectrum(df[df.time_s >= 16]["az_g"].values[:1600], FS)
from scipy.signal import find_peaks
for name, f, a in [("정상", f_n, a_n), ("이상", f_f, a_f)]:
    pk = find_peaks(a, height=0.02)[0]
    top = pk[np.argsort(a[pk])[::-1][:3]]
    print(f"{name} 구간 주요 주파수: " + ", ".join(f"{f[j]:.0f}Hz({a[j]:.2f}g)" for j in top))

## 8. 시각화

In [ ]:
fig, ax = plt.subplots(3, 1, figsize=(10, 8))
ax[0].plot(df.time_s, df.az_g, "k-", lw=0.3); ax[0].set_ylabel("Z축 (g)"); ax[0].set_title("ADXL345 모터 진동 – 원시 파형")
ax[1].plot(rms.time_s, rms.rms_g, "k-o", ms=3); ax[1].axhline(TH, color="gray", ls="--", label=f"기준 {TH}g")
ax[1].set_ylabel("RMS (g)"); ax[1].set_xlabel("시간 (초)"); ax[1].legend()
ax[2].plot(f_n, a_n, color="gray", lw=1.2, label="정상 (0~5초)"); ax[2].plot(f_f, a_f, "k-", lw=1.2, label="이상 (16~20초)")
ax[2].set_xlim(0, 200); ax[2].set_xlabel("주파수 (Hz)"); ax[2].set_ylabel("진폭 (g)"); ax[2].legend()
for a in ax: a.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 9. 저장과 IoT 전송
측정값은 CSV 파일로 저장하고, IoT 서버로 보낼 때는 보통 **JSON** 형식의 메시지를 **MQTT** 프로토콜로 전송합니다.
(Colab에서는 왼쪽 📁 메뉴에서 저장된 CSV를 내려받을 수 있습니다.)

In [ ]:
csv_name = "진동센서_data.csv"
df.to_csv(csv_name, index=False, encoding="utf-8-sig")
print("CSV 저장:", csv_name)

def to_jsonable(v):
    if isinstance(v, (bool, np.bool_)): return bool(v)
    if isinstance(v, (int, np.integer)): return int(v)
    if isinstance(v, (float, np.floating)): return None if np.isnan(v) else round(float(v), 3)
    return str(v)

last = df.iloc[-1].to_dict()
payload = json.dumps({"device": "rpi-01", "sensor": "ADXL345 3축 가속도 센서",
                      "time": datetime.now().isoformat(timespec="seconds"),
                      "values": {k: to_jsonable(v) for k, v in last.items()}}, ensure_ascii=False)
print("IoT 서버로 보낼 메시지(JSON):")
print(payload)

# ▼ 실제로 MQTT 브로커에 보내려면 (pip install paho-mqtt) 주석을 해제하세요.
# import paho.mqtt.client as mqtt
# client = mqtt.Client()
# client.connect("broker.hivemq.com", 1883)          # 공개 테스트 브로커
# client.publish("myclass/rpi-01/vibration", payload)
# client.disconnect()

## 직접 해 보기
- 측정 속도를 100Hz(`INTERVAL = 0.01`)로 낮추면 120Hz 성분이 FFT에서 어떻게 보이나요? (나이퀴스트)
- 모터 속도를 3000rpm으로 바꾸면 회전 주파수는 몇 Hz가 되나요? 가상 센서를 고쳐 확인해 보세요.
- RMS가 3구간 연속 기준을 넘을 때만 경보를 내도록 바꿔 보세요.